# Pluto M2 on Kaggle: first voice (~30M speech+text, TinyStories read aloud)

Stages: **0** sanity-check the off-the-shelf speech tools, **1** build the synthetic speech dataset,
**2** text-only warm start of the brain, **3a/3b** aligned vs text-then-audio training,
**4** evaluation (Whisper WER), demo audio and report.

Settings → **Accelerator: GPU (T4 ×2 or P100)**, **Internet: on**.

Add-ons → Secrets (never paste tokens into cells):
- `HF_TOKEN` (**strongly recommended**): a Hugging Face *write* token. Checkpoints, logs **and the speech
  dataset** are synced to a *private* Hub repo, so every stage survives the session being killed.
- `GITHUB_TOKEN`: needed while the GitHub repo is private, and for the final push cell.

Off-the-shelf components (disclosed): Kokoro-82M TTS (data synthesis), Kyutai Mimi codec
(audio tokens), Whisper small.en (QA filtering and WER scoring). The language model is ours, trained from scratch.

In [ ]:
import os, subprocess
REPO = "thedeepakreddy/Pluto-LLM"
BRANCH = "main"
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
except Exception:
    secrets = None

def secret(name):
    try:
        return secrets.get_secret(name) if secrets else os.environ.get(name)
    except Exception:
        return None

gh = secret("GITHUB_TOKEN")
url = f"https://{gh}@github.com/{REPO}.git" if gh else f"https://github.com/{REPO}.git"
if not os.path.exists("Pluto-LLM"):
    subprocess.run(["git", "clone", "-b", BRANCH, url, "Pluto-LLM"], check=True)
%cd Pluto-LLM
!apt-get install -y -q espeak-ng
# Kokoro (TTS) needs Python <= 3.12, but Kaggle runs 3.13: run the pipeline in a 3.12 env.
!pip install -q uv
!uv venv -q --python 3.12 /kaggle/working/py312
!uv pip install -q --python /kaggle/working/py312/bin/python -e ".[speech,hub]"
LLM = "/kaggle/working/py312/bin/llm"   # every `!{LLM} ...` command uses the 3.12 env
!{LLM} --help > /dev/null && echo "llm CLI ready (Python 3.12)"
!pip install -q -e ".[hub]"   # light install for the few in-notebook Python cells (no speech deps)
import sys, importlib
# An editable install is only visible to *new* processes; make it importable in this kernel too.
sys.path.insert(0, os.path.abspath("src"))
importlib.invalidate_caches()

hf = secret("HF_TOKEN")
HUB = ""
if hf:
    os.environ["HF_TOKEN"] = hf
    from huggingface_hub import HfApi
    HUB = HfApi().whoami()["name"] + "/pluto-runs"   # private repo, created on first upload
print("Hub sync:", HUB or "OFF (set the HF_TOKEN secret to survive session kills)")
SYNC = f"--set train.hub_repo={HUB}" if HUB else ""

## Creator identity (optional, private)

Leave empty to train without it. If set, it must be a **private** Hugging Face dataset that holds your
creator-identity file; it is never committed and never uploaded anywhere public.

In [ ]:
IDENTITY_DATASET = "thedeepakreddy/pluto-creator-identity"  # e.g. "your-username/pluto-creator-identity" (private)
ID = f"--set identity.hf_dataset={IDENTITY_DATASET} --set identity.speech_every=100" if IDENTITY_DATASET else ""
print("identity:", IDENTITY_DATASET or "OFF")

## Speech-data sync helpers

The speech dataset takes many hours to build, so it is uploaded to the private Hub repo under `speech/`
after every build session and downloaded at the start of a new session.

In [ ]:
SPEECH_DIR = "speech/tinystories-speech-v1"

def pull_speech():
    if not HUB:
        print("Hub sync is OFF (HF_TOKEN secret not attached): speech data will NOT survive a session restart")
        return
    from huggingface_hub import HfApi, snapshot_download
    api = HfApi()
    if not api.repo_exists(HUB) or not any(f.startswith(SPEECH_DIR + "/") for f in api.list_repo_files(HUB)):
        print("no speech data on the Hub yet")
        return
    snapshot_download(HUB, allow_patterns=[SPEECH_DIR + "/*"], local_dir=".")
    print("pulled", SPEECH_DIR)

def push_speech():
    if not HUB:
        print("Hub sync is OFF: speech data not uploaded")
        return
    if not os.path.isdir(SPEECH_DIR):
        print("no speech data to push yet")
        return
    from huggingface_hub import HfApi
    api = HfApi()
    api.create_repo(HUB, private=True, exist_ok=True)   # private: may contain identity audio
    api.upload_folder(repo_id=HUB, folder_path=SPEECH_DIR, path_in_repo=SPEECH_DIR,
                      commit_message="speech data sync")
    print("pushed", SPEECH_DIR)

pull_speech()

## Text data and tokenizer (reused from M1)
Same TinyStories data, tokenizer and shards as M1 (`data/tokenizer/shards` sections are identical).
Re-running gives identical artifacts; expect roughly 15–40 minutes on a fresh session.

In [ ]:
WARM = "configs/m2-warmstart.yaml"
ALIGNED = "configs/m2-speech-aligned.yaml"
BASELINE = "configs/m2-speech-baseline.yaml"
STAGE0 = "configs/m2-stage0.yaml"
!{LLM} prepare-data {WARM}
!{LLM} train-tokenizer {WARM}
!{LLM} tokenize {WARM}
if IDENTITY_DATASET:
    !{LLM} prepare-identity {ALIGNED} {ID}

## Stage 0: speech-check
Synthesizes a few real val chunks with Kokoro, round-trips them through Mimi and scores them with
Whisper. Read the printed report before spending hours on Stage 1.

In [ ]:
!{LLM} speech-check {STAGE0} --n 20

## Stage 1: build the speech dataset (resumable, multi-session)
Each session builds for at most `--max-hours-session` hours, then the data is pushed to the Hub.
Re-run this cell (in new sessions if needed) until it prints `complete`.
If Kokoro's word timings fail, use `--timings whisper` instead.

In [ ]:
!{LLM} build-speech {ALIGNED} {ID} --max-hours-session 9
push_speech()

## Stage 2: text-only warm start
Trains the M2 brain (8 layers, d 384, ctx 640) on text, then copies its best checkpoint to the
`audio.init_from` path used by both speech configs.

In [ ]:
!{LLM} train {WARM} {SYNC}
import glob, shutil
WARM_RUN = sorted(glob.glob("runs/m2-warmstart-*"))[-1]
print(WARM_RUN)

In [ ]:
# After a session restart: re-run setup cells 1-7 first (install, secrets, prepare-identity) — resume and eval need the identity files they create.
# If the session died: set WARM_RUN_ID and run this cell (pulls from the Hub, resumes, copies best.pt).
# WARM_RUN_ID = "m2-warmstart-..."
# WARM_RUN = f"runs/{WARM_RUN_ID}"
if not os.path.exists(WARM_RUN) and HUB:
    !{LLM} pull-run {HUB} {os.path.basename(WARM_RUN)}
!{LLM} resume {WARM_RUN}
os.makedirs("checkpoints", exist_ok=True)
shutil.copy(f"{WARM_RUN}/checkpoints/best.pt", "checkpoints/m2-warmstart-best.pt")

## Stage 3a: aligned speech model

In [ ]:
!{LLM} train {ALIGNED} {SYNC} {ID}
ALIGNED_RUN = sorted(glob.glob("runs/m2-aligned-*"))[-1]
print(ALIGNED_RUN)

In [ ]:
# After a session restart: re-run setup cells 1-7 first (install, secrets, prepare-identity) — resume and eval need the identity files they create.
# Resume (after a timeout or session death). Needs Stage 1 data (pull_speech) and checkpoints/m2-warmstart-best.pt
# only for a fresh start, not for resuming.
# ALIGNED_RUN = "runs/m2-aligned-..."
if not os.path.exists(ALIGNED_RUN) and HUB:
    !{LLM} pull-run {HUB} {os.path.basename(ALIGNED_RUN)}
!{LLM} resume {ALIGNED_RUN}

## Stage 3b: text-then-audio baseline
Identical config except `audio.layout` (and `run.name`): same data, steps, lr and seed.

In [ ]:
!{LLM} train {BASELINE} {SYNC} {ID}
BASELINE_RUN = sorted(glob.glob("runs/m2-baseline-*"))[-1]
print(BASELINE_RUN)

In [ ]:
# After a session restart: re-run setup cells 1-7 first (install, secrets, prepare-identity) — resume and eval need the identity files they create.
# BASELINE_RUN = "runs/m2-baseline-..."
if not os.path.exists(BASELINE_RUN) and HUB:
    !{LLM} pull-run {HUB} {os.path.basename(BASELINE_RUN)}
!{LLM} resume {BASELINE_RUN}

## Stage 4: evaluation, demo audio, report
`eval-speech` scores 200 held-out test chunks with Whisper (WER/CER, duration ratio, RTF);
`--floor` also scores an untrained model so the numbers have a reference point.

In [ ]:
!{LLM} eval-speech {ALIGNED_RUN} --floor
!{LLM} eval-speech {BASELINE_RUN} --floor

In [ ]:
from IPython.display import Audio, Markdown, display
from llm.speech.evaluate import DEMO_SENTENCES
for run in (ALIGNED_RUN, BASELINE_RUN):
    os.makedirs(f"{run}/demo", exist_ok=True)
    for i, text in enumerate(DEMO_SENTENCES[:3]):
        wav = f"{run}/demo/demo_{i}.wav"
        !{LLM} speak {run} --text "{text}" --out {wav} --seed 1
        print(run, "|", text)
        display(Audio(wav))

In [ ]:
for run in (ALIGNED_RUN, BASELINE_RUN):
    open(f"{run}/limitations.md", "w").write(
        "# Known limitations\n\n"
        "- ~30M parameters; speech is synthetic (one Kokoro voice) reading TinyStories only.\n"
        "- Audio quality is bounded by the Mimi codec; WER is measured by Whisper, itself imperfect.\n"
        "- Single run per layout, single seed.\n")
    !{LLM} report {run}
!{LLM} compare {ALIGNED_RUN} {BASELINE_RUN}
display(Markdown(open(f"{ALIGNED_RUN}/report.md").read().replace("](charts/", f"]({ALIGNED_RUN}/charts/")))

## Push results to GitHub (public artifacts only; identity redacted by `export_public_results`)

In [ ]:
import shutil, subprocess
def run_git(*args):
    r = subprocess.run(["git", *args], capture_output=True, text=True)
    if r.returncode != 0:
        msg = (r.stderr + r.stdout).replace(gh, "***") if gh else r.stderr + r.stdout
        raise RuntimeError(f"git {args[0]} failed: {msg}")
    return r.stdout

if not gh:
    print("GITHUB_TOKEN not set; skipping push")
else:
    from llm.publish import export_public_results   # redacts creator identity; raises on any leak
    for run in (ALIGNED_RUN, BASELINE_RUN):
        export_public_results(run, f"results/m2/{os.path.basename(run)}")
    run_git("config", "user.name", "Pluto Kaggle")
    run_git("config", "user.email", "kaggle@users.noreply.github.com")
    run_git("add", "results")
    if not run_git("status", "--porcelain", "results").strip():
        print("nothing changed; not committing")
    else:
        run_git("commit", "-m", "results(m2): Kaggle run reports")
        run_git("pull", "--rebase", url, BRANCH)
        run_git("push", url, f"HEAD:{BRANCH}")
        print("pushed results/")